# Задание 1. MNIST: первые 10 цифр тестовой выборки
Столбец 1 — изображение, столбец 2 — что распознала нейросеть, столбец 3 — что изображено на самом деле.

In [ ]:
# === Ячейка 1: загрузка данных и обучение ===
import numpy as np                 # работа с массивами (argmax)
import matplotlib.pyplot as plt    # рисование картинок и таблицы
import keras                       # Keras (в Colab уже установлен)

# Загружаем MNIST: 60 000 обучающих и 10 000 тестовых картинок 28x28
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Нормализация: пиксели 0..255 -> 0..1 (сети так учатся лучше)
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Простая полносвязная сеть
model = keras.Sequential([
    keras.Input(shape=(28, 28)),                    # на входе картинка 28x28
    keras.layers.Flatten(),                         # вытягиваем в вектор из 784 чисел
    keras.layers.Dense(128, activation="relu"),     # скрытый слой из 128 нейронов
    keras.layers.Dense(10, activation="softmax"),   # 10 выходов = вероятности цифр 0..9
])

# Настраиваем обучение: оптимизатор Adam, функция потерь для меток-чисел, метрика точность
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(x_train, y_train, epochs=5)   # 5 проходов по обучающим данным
model.evaluate(x_test, y_test)          # точность на тесте (ожидаемо около 0.97-0.98)

In [ ]:
# === Ячейка 2: предсказания для первых 10 тестовых цифр ===
probs = model.predict(x_test[:10])   # форма (10, 10): 10 вероятностей для каждой картинки
pred = np.argmax(probs, axis=1)      # что распознала сеть = номер максимальной вероятности
true = y_test[:10]                   # что изображено на самом деле (правильные метки)
print("Сеть:  ", pred)
print("Правда:", true)

In [ ]:
# === Ячейка 3: таблица 10 строк x 3 столбца ===
fig, axes = plt.subplots(10, 3, figsize=(6, 16))   # сетка 10x3, размер в дюймах

# Заголовки столбцов (над первой строкой)
for col, name in enumerate(["Изображение", "Распознала сеть", "На самом деле"]):
    axes[0, col].set_title(name, fontsize=12)

for i in range(10):
    ok = pred[i] == true[i]                   # угадала ли сеть
    color = "green" if ok else "red"          # ошибку выделяем красным

    axes[i, 0].imshow(x_test[i], cmap="gray") # столбец 1: сама картинка
    axes[i, 1].text(0.5, 0.5, str(pred[i]),   # столбец 2: ответ сети
                    fontsize=28, ha="center", va="center", color=color)
    axes[i, 2].text(0.5, 0.5, str(true[i]),   # столбец 3: правильный ответ
                    fontsize=28, ha="center", va="center")

    for ax in axes[i]:
        ax.axis("off")                        # убираем рамки и оси

plt.tight_layout()   # аккуратно раскладываем ячейки без наложений
plt.show()